<a href="https://colab.research.google.com/github/toplubster/Biocoding-Stuffs-/blob/main/da_supa_conc_calc_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Da Supa Conc Calc (2.0)

## **Purpose:** This was created to assist with data plotting and data calculations for BCA protein concentration assay



---



###**Default Values (Chart title, axis titles, units)**

``assay_name       = 'Protein Assay'``

``wavelength       = 562 (nm)``

``conc_units       = 'mg/mL'  ``

``concentration_col = 'Concentration_mg_per_mL'``

``replicate_cols   = ['Rep1', 'Rep2']``


> If you would like to modify any of these, you will have to modify the code cell in **step 1**.



> If you have any questions, please reach out to me at hieuu.nguyen2511@gmail.com ginopark41@gmail.com
or
Professor Novak at novakw@wabash.edu






# Step 1: Importing Libraries and Data Acquisition

In [ ]:
#@title Run this cell to import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import urllib.request
import os

print("✅ All libraries imported successfully!")

# ─────────────────────────────────────────────────────────
# 🔧 TEMPLATE MODIFICATION POINT — Change these five variables
# ─────────────────────────────────────────────────────────

assay_name       = 'BCA Protein Assay'   # 🔧 Your assay name
wavelength       = 562                         # 🔧 Measurement wavelength (nm)
conc_units       = 'mg/mL'                    # 🔧 Concentration units
concentration_col = 'Concentration_mg_per_mL' # 🔧 First column name in your CSV
replicate_cols   = ['Rep1', 'Rep2']   # 🔧 Replicate column names

# ─────────────────────────────────────────────────────────
# No changes needed below this line
# ─────────────────────────────────────────────────────────
print(f"✅ Parameters set:")
print(f"   Assay:       {assay_name}")
print(f"   Wavelength:  {wavelength} nm")
print(f"   Units:       {conc_units}")
print(f"   Conc column: {concentration_col}")
print(f"   Replicates:  {replicate_cols}")

# Step 2: Uploading Data and Checking Data


> 🔧 ** IMPORTANT NOTE**
> Your data must have:
> - One column of **concentration values** (your x-axis)
> - One or more columns of **replicate measurements** (your y-axis readings)
> - Column names that **exactly match** `concentration_col`
>   and `replicate_cols` defined in Section 3a
> - All of your samples name column
> - One or more Unknown Replicates column
> - Dilution for each sample


> ## **TL;DR:** Make sure your .csv file format is exactly as the example below or I will crash out.



---



#### What Your CSV Must Look Like

Your CSV file must have the following structure:

| Concentration_mg_per_mL | Rep1 | Rep2 | Sample | Unk_rep1 | Unk_rep2 | Dilution |
|---|---|---|---|---|---|---|
| 0 | 0.1049 | 0.1048 | N2 | 0.4319 | 0.4268 | 25 |
| 0.025 | 0.1149 | 0.1201 | N2 40 | 0.2533 | 0.2564 | 40 |
| 0.125 | 0.1672 | 0.1676 | N2 50 | 0.1916 | 0.1924 | 50 |
| 0.25 | 0.2344 | 0.2312 | lon 25 | 0.3434 | 0.3319 | 25 |
| 0.5 | 0.3528 | 0.3441 | lon 40 | 0.2786 | 0.2586 | 40 |
| 0.75 | 0.5133 | 0.3801 | lon 50 | 0.3227 | 0.2306 | 50 |
| 1 | 0.5203 | 0.5114 | svh 25 | 0.2663 | 0.2707 | 25 |
| 1.5 | 0.8098 | 0.761 | svh 40 | 0.2184 | 0.226 | 40 |
| 2 | 1.0645 | 0.9611 | svh 50 | 0.2076 | 0.1864 | 50 |
| | | | con 25 | 0.4348 | 0.425 | 25 |
| | | | con 40 | 0.252 | 0.2483 | 40 |
| | | | con 50 | 0.3505 | 0.309 | 50 |
| | | | K12 25 | 0.376 | 0.4021 | 25 |
| | | | K12 40 | 0.2195 | 0.2405 | 40 |
| | | | K12 50 | 0.2309 | 0.2683 | 50 |
| | | | tpp 25 | 0.5013 | 0.5081 | 25 |
| | | | tpp 40 | 0.3444 | 0.3365 | 40 |
| | | | tpp 50 | 0.2595 | 0.296 | 50 |

**Rules:**
- The **first row** must be a header row with column names
- The **first column** must contain your concentration values
- The **remaining columns** must contain replicate measurements
- Column names must **exactly match** `concentration_col` and
  `replicate_cols` defined above.
- Please refer to the example above to make an appropriate `.csv` file (All **column names** must match!
- Values must be **numbers only** except for sample names— no units in the data cells
- Save the file as `.csv` (Comma Separated Values)

> **In Excel or Google Sheets:**
> File → Download → Comma Separated Values (.csv)

---

#### How to Upload to Colab

There are two ways:

**Method 1 — Drag and drop:**
1. Click the **folder icon (📁)** in the left panel
2. Drag your CSV file from your computer into the panel

**Method 2 — Upload dialog (used in the code below):**
1. Run the cell below
2. Click **Choose Files** in the dialog that appears
3. Navigate to and select your CSV file
4. Wait for the upload to complete









> ## **You will only need to use either one of the methods**

In [ ]:
#@title **Method 1:** Upload your own CSV file from your computer


from google.colab import files

print("A file chooser dialog will appear below.")
print("Select your CSV file and wait for the upload to complete.")
print()

# This opens the file upload dialog
uploaded = files.upload()

# 🔧 If your file has a different name, update it here
# The filename is shown after upload completes
uploaded_filename = list(uploaded.keys())[0]

print(f"\n✅ File uploaded: {uploaded_filename}")

# Read the uploaded file into a DataFrame
df = pd.read_csv(uploaded_filename)

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print()
print("Column names found in your file:")
print(list(df.columns))
print()
print("⚠️  Check that column names match concentration_col and")
print(f"   replicate_cols defined in Section 3a:")
print(f"   concentration_col = '{concentration_col}'")
print(f"   replicate_cols    = {replicate_cols}")

# Verify your data — run this regardless of which option you used
'''
Check that:
- The **shape** matches the number of rows and columns you expect
- The **column names** match what you defined in Section 3a
- The **data types** are numeric (`float64` or `int64`) for all columns, sample column data type can be different
- The **head** shows sensible values — no obvious errors
'''

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print()
print("Column names:")
print(list(df.columns))
print()
print("Data types:")
print(df.dtypes)

# Replace NaN values with blank strings for display
display(df.fillna(''))

In [ ]:
#@title **Method 2:** Option to load CSV directly from the Colab notebook folder by typing the filename
import pandas as pd

print("Enter the filename of your CSV file (e.g., 'my_data.csv'):")
filename = input()

try:
    # Read the specified file into a DataFrame
    df = pd.read_csv(filename)

    print(f"\n✅ File loaded: {filename}")
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print()
    print("Column names found in your file:")
    print(list(df.columns))
    print()
    print("⚠️  Check that column names match concentration_col and")
    print(f"   replicate_cols defined in Section 3a:")
    print(f"   concentration_col = '{concentration_col}'")
    print(f"   replicate_cols    = {replicate_cols}")

    # Verify your data
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print()
    print("Column names:")
    print(list(df.columns))
    print()
    print("Data types:")
    print(df.dtypes)

    # Replace NaN values with blank strings for display
    display(df.fillna(''))

except FileNotFoundError:
    print(f"Error: The file '{filename}' was not found in your Colab environment. Please ensure the file name is correct and it is in the /content/ folder.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

#Step 3: Calculations
## 3.1 Calculating the Average and Standard Deviation

Each concentration point was measured multiple times (one reading
per replicate column). We collapse these into:
- **Average** — the central value we will plot and fit
- **Standard deviation** — the spread we will show as error bars

In [ ]:
#@title Calculate average and standard deviation across replicates
# Uses replicate_cols defined in Section 3a — no changes needed here

# Filter df to include only standard curve points
std_data_df = df[df[concentration_col].notna()].copy()

# Average across replicate columns for each row
std_data_df['average'] = std_data_df[replicate_cols].mean(axis=1)

# Standard deviation across replicate columns for each row
std_data_df['std_dev'] = std_data_df[replicate_cols].std(axis=1)

# Zeroing the absorbance relative to the first (blank) standard
std_data_df['zeroed'] = std_data_df['average'] - std_data_df['average'].iloc[0]

# Display only the specified columns in the desired order
display(std_data_df[[concentration_col, 'Rep1', 'Rep2', 'average', 'std_dev', 'zeroed']])



## 3.2 Fitting the Curve: Nonlinear Regression



In [ ]:
#@title Plotting and Finding The Best Fit Curve
import numpy as np # Ensure numpy is imported

# Step 1: Store the full result object and explore it

# Use the filtered standard curve data for regression
concentration      = std_data_df[concentration_col]
average_absorbance = std_data_df['zeroed']

# Use numpy.polyfit to find the coefficients for a quadratic model: y = ax² + bx + c
# popt will contain [a, b, c]
popt = np.polyfit(concentration, average_absorbance, 2)

# Unpack parameters
a, b, c = popt

# Define a function to evaluate the polynomial for residuals and plotting
def model_poly(x, popt_coeffs):
    return np.polyval(popt_coeffs, x)

# Calculate R² manually from residuals
residuals = average_absorbance - model_poly(concentration, popt)
ss_res    = np.sum(residuals**2)
ss_tot    = np.sum((average_absorbance - np.mean(average_absorbance))**2)
r_squared = 1 - (ss_res / ss_tot)

# Print a clean summary
print(f"{assay_name} — Polynomial Regression (Degree 2) Results")
print("=" * 45)
print(f"  a (quadratic): {a:.6f}")
print(f"  b (linear):    {b:.4f}")
print(f"  c (intercept): {c:.4f}")
print(f"  R²:            {r_squared:.4f}")
print()

# Evaluate fit quality
if r_squared >= 0.99:
    print("Fit quality: Excellent ✅")
elif r_squared >= 0.95:
    print("Fit quality: Acceptable ⚠️")
else:
    print("Fit quality: Check your data ❌")

# Plot with best-fit curve

# Generate 100 evenly-spaced points for a smooth curve
curve_x = np.linspace(concentration.min(), concentration.max(), 100)
curve_y = model_poly(curve_x, popt)

plt.figure(figsize=(8, 6))

# Error bar data points
plt.errorbar(
    concentration,
    average_absorbance,
    yerr=std_data_df['std_dev'], # Use std_data_df['std_dev'] for correct length
    fmt='o',
    capsize=4,
    label='Average ± Std Dev'
)

# Best-fit curve
plt.plot(
    curve_x, curve_y,
    color='red',
    label=f'Best-fit: y = {a:.4f}x² + {b:.4f}x + {c:.4f}\nR² = {r_squared:.4f}'
)

plt.xlabel(f'Concentration ({conc_units})')
plt.ylabel(f'Absorbance at {wavelength} nm')
plt.title(f'{assay_name} Standard Curve with Best-fit Curve')
plt.grid(True)
plt.legend()
plt.show()

> ### If fit quality is "Check your data ❌," don't panic. Try to look at the .csv file or your data again.



### Predicting Concentration from Absorbance



In [ ]:
#@title Zero the samples absorbance and calculating concentrations.
import numpy as np
import pandas as pd # Ensure pandas is imported for pd.notna

# Define the function to convert absorbance to concentration
def solve_concentration(y, a, b, c):
    A = a
    B = b
    C = c - y # 'y' here is the value passed into the function (unk_average in this case)

    discriminant = B**2 - 4*A*C

    if discriminant < 0:
        print(f"DEBUG: y={y:.5f}, Discriminant < 0: {discriminant:.2e}") # Diagnostic print
        return np.nan

    root1 = (-B + np.sqrt(discriminant)) / (2*A)
    root2 = (-B - np.sqrt(discriminant)) / (2*A)

    roots = [r for r in [root1, root2] if r >= 0]

    if len(roots) == 0:
        print(f"DEBUG: y={y:.5f}, No positive roots found. Roots: {root1:.2e}, {root2:.2e}") # Diagnostic print
        return np.nan

    # Choose the maximum positive root, which is typically the physically relevant one
    return max(roots)

# --- Start of added code to recreate df_unk ---
# Create df_unk from the full DataFrame, including all rows that have unknown replicate data.
# We assume that any row with a non-NaN value in 'Unk_rep1' or 'Unk_rep2' is a sample to be processed.
unknown_replicate_cols = ['Unk_rep1', 'Unk_rep2']

df_unk = df[df[unknown_replicate_cols[0]].notna() | df[unknown_replicate_cols[1]].notna()].copy()

# Drop rows where all unknown replicate values are NaN
df_unk = df_unk.dropna(subset=unknown_replicate_cols)

# Calculate the average of unknown replicates
df_unk['Unk_Average_Abs'] = df_unk[unknown_replicate_cols].mean(axis=1)

# Zero the unknown average absorbance relative to the blank from std_data_df
# Ensure std_data_df is available and contains the 'average' column (from the blank)
if 'average' in std_data_df.columns and len(std_data_df) > 0:
    blank_average = std_data_df['average'].iloc[0]
else:
    print("Warning: Blank average not found in std_data_df. Assuming blank = 0 for unknown samples.")
    blank_average = 0
df_unk['Unk_Zeroed_Abs'] = df_unk['Unk_Average_Abs'] - blank_average
# --- End of added code to recreate df_unk ---

# 2. Measured concentration (from zeroed absorbance of unknown samples)
# Note: The coefficients a, b, c were derived from *zeroed* absorbances from the standard curve.
# We are using the 'Unk_Zeroed_Abs' column from df_unk, as calculated in the previous cell.
df_unk['measured_conc'] = df_unk['Unk_Zeroed_Abs'].apply(lambda y_raw: solve_concentration(y_raw, a, b, c))

# 3. Correct for dilution using the 'Dilution' column from df_unk
df_unk['corrected_conc'] = df_unk['measured_conc'] * df_unk['Dilution']

# 4. Predicted absorbance from back-calculated concentration
# This uses the measured_conc derived from the quadratic equation
df_unk['predicted_absorbance'] = (
    a * df_unk['measured_conc']**2 +
    b * df_unk['measured_conc'] +
    c
)

# 5. Residual error (in absorbance space)
# This will be the difference between the original Unk_Zeroed_Abs and the predicted absorbance
# based on the calculated measured_conc.
df_unk['residual_error'] = df_unk['Unk_Zeroed_Abs'] - df_unk['predicted_absorbance']

# 6. Final table for display
# Select specific columns for the consolidated table.
result_table_cols = [
    'Sample',
    'Unk_rep1',
    'Unk_rep2',
    'Unk_Zeroed_Abs',
    'Dilution',
    'measured_conc',
    'corrected_conc',
    'residual_error'
]
result_table = df_unk[result_table_cols].copy()

# Rename columns for clarity in the final table
result_table.rename(columns={
    'Sample': 'Sample_Name',
    'Unk_rep1': 'Rep1',
    'Unk_rep2': 'Rep2',
    'Unk_Zeroed_Abs': f'Unk_Zeroed_Abs ({wavelength} nm)', # Added units
    'Dilution': 'Dilution_Factor',
    'measured_conc': f'Measured_Concentration ({conc_units})', # Added units
    'corrected_conc': f'Corrected_Concentration ({conc_units})',
    'residual_error': 'Residual_Error_Absorbance'
}, inplace=True)

# Format Residual_Error_Absorbance to scientific notation
result_table['Residual_Error_Absorbance'] = result_table['Residual_Error_Absorbance'].apply(lambda x: f"{x:.2e}" if pd.notna(x) else np.nan)

print("Calculated Concentrations and Residual Errors for All Samples (using Unk_Zeroed_Abs):")
display(result_table.round(6))

# Step 4: Exporting A CSV File for Samples
- The CSV file will be similar to the *Predicting Concentration from Absorbance* table

In [ ]:
#@title Exporting the 'Predicting Concentration from Absorbance' table to a CSV file

print("Enter the filename for the exported CSV (e.g., 'unknown_concentrations'):")
export_filename_base = input()

# Ensure the filename ends with .csv
if not export_filename_base.endswith('.csv'):
    export_filename = export_filename_base + '.csv'
else:
    export_filename = export_filename_base

try:
    # Save the df_unk DataFrame to a CSV file
    # index=False prevents pandas from writing the DataFrame index as a column in the CSV
    df_unk.to_csv(export_filename, index=False)
    print(f"\n✅ Successfully exported '{export_filename}'")
    print("You can find this file in the Colab file browser (folder icon 📁 on the left). (Hit the refresh button🔄️))")
except Exception as e:
    print(f"An error occurred while exporting the file: {e}")